# Full-Fidelity Resolved-System Review Notebook

Interactive diagnostic notebook for the canonical `full_fidelity_binary_iterative` campaign schema, using the review-scale YAML config. Run cells one at a time to inspect the resolved truth/data and inference/reference systems before launching any larger campaign.

This notebook does not run a production campaign or optimization; it uses the same wrapper translator/model-split path as the campaign runner.


## 0. Setup and configuration

Set toggles here. The path resolver supports launching from the repository root or from `examples/notebooks`.


In [ ]:
import os
import sys
import tempfile
from pathlib import Path

import jax
from IPython.display import display as notebook_display

jax.config.update("jax_enable_x64", True)


def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for q in (p, *p.parents):
        if (q / "pyproject.toml").exists() or (q / ".git").exists():
            return q
    return p

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "dluxshera-matplotlib"))
REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

CONFIG_PATH = REPO_ROOT / "examples/recipes/full_fidelity_algorithm_campaign_template/full_fidelity_binary_iterative_review.yaml"
OUTPUT_ROOT = REPO_ROOT / "Results/full_fidelity_resolved_system_review"
RUN_LABEL = "interactive_review"
WRITE_ARTIFACTS = True
FAST_RENDER = True
ENABLE_NOISE_AUDIT = True
NOISE_REVIEW_MIN_PSF_NPIX = 160
NOISE_REVIEW_DEFAULT_PSF_NPIX = 256
NOISE_REVIEW_DISPLAY_CROP_NPIX = 160  # None displays the full rendered image

print("repo root:", REPO_ROOT)
print("PYTHONPATH has src:", str(SRC) in sys.path)
print("config path:", CONFIG_PATH)
print("output root:", OUTPUT_ROOT / RUN_LABEL)


### Plotting backend policy

For interactive pan/zoom plots in Jupyter Lab, install `ipympl` and set `PLOT_BACKEND = "widget"`. If `ipympl` is unavailable, use `PLOT_BACKEND = "inline"` for reliable static plots. Use `PLOT_BACKEND = "agg"` only for headless artifact generation.


In [ ]:
# Plotting backend policy
#
# Use "widget" for interactive pan/zoom in Jupyter Lab when ipympl is installed.
# Use "inline" for robust static plots in any notebook.
# Use "auto" to prefer widget and fall back to inline.
# Do not use "agg" in this review notebook; it disables inline display.
PLOT_BACKEND = "inline"  # options: "auto", "widget", "inline", "notebook"

import importlib.util
import sys


def configure_notebook_matplotlib_backend(mode="auto"):
    """Configure Matplotlib for interactive notebook review."""
    try:
        ip = get_ipython()
    except NameError:
        ip = None

    if mode == "agg":
        raise ValueError("PLOT_BACKEND='agg' disables notebook display; use inline or widget here.")

    if ip is None:
        # Not running in IPython/Jupyter. Leave backend alone.
        return "unchanged_non_ipython"

    if mode in {"auto", "widget"}:
        if importlib.util.find_spec("ipympl") is not None:
            active = "widget"
            ip.run_line_magic("matplotlib", active)
            _switch_existing_pyplot_backend(active)
            return active
        if mode == "widget":
            print("ipympl is not installed; falling back to inline backend.")

    if mode == "notebook":
        active = "notebook"
        ip.run_line_magic("matplotlib", active)
        _switch_existing_pyplot_backend(active)
        return active

    active = "inline"
    ip.run_line_magic("matplotlib", active)
    _switch_existing_pyplot_backend(active)
    return active


def _switch_existing_pyplot_backend(mode):
    """Repair pyplot if a prior cell or stale imported helper switched to Agg."""
    if "matplotlib.pyplot" not in sys.modules:
        return
    import matplotlib.pyplot as plt

    backend = {
        "inline": "module://matplotlib_inline.backend_inline",
        "widget": "module://ipympl.backend_nbagg",
        "notebook": "nbAgg",
    }.get(mode)
    if backend is None:
        return
    try:
        plt.switch_backend(backend)
        plt.ion()
    except Exception as exc:
        print(f"Could not switch existing pyplot backend to {mode}: {exc}")


ACTIVE_MATPLOTLIB_BACKEND = configure_notebook_matplotlib_backend(PLOT_BACKEND)
print(f"Matplotlib notebook backend: {ACTIVE_MATPLOTLIB_BACKEND}")


In [ ]:
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.ion()

from dluxshera.utils import full_fidelity_review as review

cfg = review.load_smoke_config(CONFIG_PATH)
exp = cfg["experiment"]
outdir = OUTPUT_ROOT / RUN_LABEL
outdir.mkdir(parents=True, exist_ok=True)

print("schema_version:", exp.get("schema_version"))
print("run_name:", exp.get("run_name"))
print("source_kind:", exp.get("source_kind"))
print("target:", exp.get("target"))
print("system_preset:", exp.get("system_preset"))
print("output directory:", outdir)


In [ ]:
import matplotlib

probe_fig = plt.figure(figsize=(1, 1))
print("Matplotlib backend:", matplotlib.get_backend())
print("Figure canvas:", type(probe_fig.canvas).__module__, type(probe_fig.canvas).__name__)
print("Interactive mode:", plt.isinteractive())
plt.close(probe_fig)


## 0a. Config contract registry

Display registry-backed enum/option validation for the active config.


In [ ]:
from dluxshera.utils.full_fidelity_config_schema import registry_entry_for_path, iter_string_fields, validate_config_contract

contract = validate_config_contract(cfg, config_tier='review', strict=False)
print('contract findings:', len(contract['findings']))
for finding in contract['findings']:
    print(f"{finding['severity']} {finding['field_path']} {finding['code']}: {finding['message']}")

rows = []
for path, value in iter_string_fields(cfg):
    pattern, entry = registry_entry_for_path(path)
    rows.append({
        'field_path': path,
        'value': value,
        'registry_pattern': pattern,
        'valid_values': ', '.join((entry or {}).get('valid_values', {}).keys()),
        'implemented_status': (entry or {}).get('implemented_status'),
    })
notebook_display(pd.DataFrame(rows))


## 1. Translate smoke config and build model split

This uses the smoke wrapper's private translator by file-path import, then builds the package `CampaignModelSplit` without launching subblock inference.


In [ ]:
ctx = review.build_model_split_from_smoke(cfg, outdir, run_label=RUN_LABEL, write_artifacts=WRITE_ARTIFACTS)
translated = ctx["translated_config"]
base_system = ctx["base_system_cfg"]
truth_system = ctx["truth_system_cfg"]
inference_system = ctx["inference_system_cfg"]
split = ctx["model_split"]
ACTIVE_MATPLOTLIB_BACKEND = configure_notebook_matplotlib_backend(PLOT_BACKEND)

print("translated kind:", translated["experiment"].get("kind"))
print("truth hash:", split.truth_config_hash)
print("inference hash:", split.inference_config_hash)
print("matched:", split.truth_config_hash == split.inference_config_hash)
print("model split components:")
print(json.dumps(split.enabled_components, indent=2))


In [ ]:
for role, system in [("base", base_system), ("truth", truth_system), ("inference", inference_system)]:
    s = review.summarize_source_config(system)
    print(f"{role} source wavelength_m / bandwidth_m / n_lambda:", s["wavelength_m"], s["bandwidth_m"], s["n_lambda"])
    print("  explicit wavelengths_m / weights / component_weights:", s["has_wavelengths_m"], s["has_weights"], s["has_component_weights"])

print("\nOriginal review config excerpt:")
print(json.dumps({k: exp.get(k) for k in ["spectral_model", "high_order_wfe", "subblocks", "iterative"]}, indent=2)[:5000])
print("\nTranslated observation-bias config excerpt:")
print(json.dumps(translated["experiment"], indent=2)[:5000])


## 2. Spectral model review

Answers: actual truth/reference wavelength grids, `fast` clamp, inference wavelength override, component SED/weight differences, response-curve availability, and flux-factor provenance.


In [ ]:
spectral_summary = review.summarize_spectral_deck(split)
spectral_tables = review.spectral_review_tables(base_system, truth_system, inference_system)
truth_spec = pd.DataFrame(spectral_tables["truth"])
inf_spec = pd.DataFrame(spectral_tables["inference"])
responses = review.response_curve_review(translated["experiment"].get("spectral_model"))

print(json.dumps({k: v for k, v in spectral_summary.items() if k != "provenance"}, indent=2, default=str))
print("truth rows:", len(truth_spec), "inference rows:", len(inf_spec))
print("detector QE enabled/available:", responses["detector_qe"]["enabled"], responses["detector_qe"]["available"])
print("M2 filter enabled/available:", responses["m2_filter_response"]["enabled"], responses["m2_filter_response"]["available"])

if WRITE_ARTIFACTS:
    review.write_spectral_review_csv(outdir / "spectral_review_tables.csv", spectral_tables)

notebook_display(truth_spec)
notebook_display(inf_spec)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, (label, response) in zip(axes[0], responses.items()):
    if response["available"]:
        ax.plot(response["wavelengths_nm"], response["response"])
    ax.set_title(f"{label}: {'active' if response['enabled'] else 'available but not active'}")
    ax.set_xlabel("wavelength [nm]")
    ax.set_ylabel("response")

for role, df, ax in [("truth", truth_spec, axes[1,0]), ("inference", inf_spec, axes[1,1])]:
    for comp, group in df.groupby("component"):
        ax.plot(group["wavelength_nm"], group["weight"], marker="o", label=comp)
    ax.set_title(f"Effective {role} component weights")
    ax.set_xlabel("wavelength [nm]")
    ax.set_ylabel("normalized sample weight")
    ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for comp in sorted(set(truth_spec["component"])):
    t = truth_spec[truth_spec["component"] == comp]
    i = inf_spec[inf_spec["component"] == comp]
    axes[0].plot(t["wavelength_nm"], t["weight"], marker="o", label=f"truth {comp}")
    axes[0].plot(i["wavelength_nm"], i["weight"], marker="s", linestyle="--", label=f"inference {comp}")

if {"primary", "secondary"} <= set(truth_spec["component"]):
    tp = truth_spec[truth_spec["component"] == "primary"].sort_values("wavelength_nm")
    ts = truth_spec[truth_spec["component"] == "secondary"].sort_values("wavelength_nm")
    axes[1].plot(tp["wavelength_nm"], tp["weight"].to_numpy() - ts["weight"].to_numpy(), marker="o", label="truth primary-secondary")
if {"primary", "secondary"} <= set(inf_spec["component"]):
    ip = inf_spec[inf_spec["component"] == "primary"].sort_values("wavelength_nm")
    isec = inf_spec[inf_spec["component"] == "secondary"].sort_values("wavelength_nm")
    axes[1].plot(ip["wavelength_nm"], ip["weight"].to_numpy() - isec["weight"].to_numpy(), marker="s", label="inference primary-secondary")
axes[0].set_title("Truth vs inference effective spectral response")
axes[0].set_xlabel("wavelength [nm]")
axes[0].set_ylabel("weight")
axes[0].legend()
axes[1].set_title("Component weight difference")
axes[1].set_xlabel("wavelength [nm]")
axes[1].set_ylabel("primary - secondary")
axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
def effective_rows(df):
    rows = []
    for comp, group in df.groupby("component"):
        w = group["weight"].to_numpy(float)
        lam = group["wavelength_nm"].to_numpy(float)
        rows.append({"component": comp, "sum_weights": w.sum(), "effective_wavelength_nm": (w * lam).sum() / w.sum()})
    return pd.DataFrame(rows)

print("Truth effective wavelengths")
notebook_display(effective_rows(truth_spec))
print("Inference effective wavelengths")
notebook_display(effective_rows(inf_spec))
print("Flux/provenance entries")
print(json.dumps(spectral_summary["provenance"], indent=2, default=str)[:8000])


## 3. Preserve-flux-parameters review

Verifies that spectral deck patching changes chromatic shape and provenance, not scalar band-integrated flux parameters, when `preserve_flux_parameters: true`.


In [ ]:
flux_review = review.preserve_flux_review(base_system, truth_system, inference_system, translated["experiment"].get("spectral_model"))
print(json.dumps(flux_review, indent=2, default=str))
for warning in flux_review["warnings"]:
    print("WARNING:", warning)

for role, system in [("truth", truth_system), ("inference", inference_system)]:
    s = review.summarize_source_config(system)
    print(role, "component row sums:", s["component_weight_sums"])
    assert all(np.isclose(v, 1.0) for v in s["component_weight_sums"]), role


## 4. High-order WFE review

This section checks the WFE bookkeeping. The raw PTT-removed OPD map is decomposed into stored low-order Zernike coefficients plus a high-order residual map. The high-order knowledge error is also projected into the high-order residual subspace so that low-order WFE uncertainty is not double-counted. The inference/reference high-order map should equal the truth high-order residual plus the high-order knowledge-error residual.

`npix=16` is smoke-only and should not be used for serious WFE studies. A practical next step is to match the generated high-order map sampling to the resolved pupil sampling (`optics.pupil_npix`) when feasible, or document a downsample/upsample policy.


In [ ]:
wfe_summary = review.summarize_wfe_artifacts(split)
print("enabled:", wfe_summary.get("enabled"))
print("warnings:", wfe_summary.get("warnings"))
for mirror, item in wfe_summary.get("mirrors", {}).items():
    print("\n", mirror)
    print(" requested truth RMS:", item["requested_truth_rms_nm"], "high-order residual measured:", item["rms_nm"]["truth_high_order_residual"])
    print(" requested error RMS:", item["requested_knowledge_error_rms_nm"], "residual measured:", item["rms_nm"]["knowledge_error_residual"])
    print(" raw = low-order + residual RMS:", item["rms_nm"]["raw_ptt_removed_truth"], item["rms_nm"]["low_order_reconstruction"], item["rms_nm"]["truth_high_order_residual"])
    print(" inference sum residual RMS:", item["rms_nm"]["inference_sum_residual"])
    print(" stored coefficient index mapping:", item["coefficient_array_index_mapping"])
    if item["warnings"]:
        print(" mirror warnings:", item["warnings"])
    if item["truth_high_order_residual_opd_nm"].shape[0] < int(truth_system["optics"].get("pupil_npix", 0)):
        print("WARNING: WFE npix is much smaller than optics pupil_npix", item["truth_high_order_residual_opd_nm"].shape, truth_system["optics"].get("pupil_npix"))


In [ ]:
wfe_cmap = review.cmap_with_bad("RdBu_r", bad="0.5")
for mirror, item in wfe_summary.get("mirrors", {}).items():
    title_prefix = mirror.capitalize()
    fig, axes = plt.subplots(2, 4, figsize=(18, 8.5))
    fig.suptitle(f"{title_prefix} WFE decomposition review", fontsize=14)
    image_panels = [
        ("Raw PTT-removed truth OPD [nm]", item["raw_ptt_removed_truth_opd_nm"]),
        ("Low-order Zernike reconstruction [nm]", item["low_order_truth_reconstruction_nm"]),
        ("Truth high-order residual OPD [nm]", item["truth_high_order_residual_opd_nm"]),
        ("High-order knowledge-error residual OPD [nm]", item["knowledge_error_high_order_residual_opd_nm"]),
        ("Inference high-order OPD [nm]", item["inference_high_order_opd_nm"]),
        ("Inference - truth residual - error [nm]", item["inference_sum_residual_nm"]),
    ]
    for ax, (title, arr) in zip(axes.flat[:6], image_panels):
        masked = review.masked_for_imshow(arr, item["mask"])
        vmin, vmax = review.symmetric_nan_limits(masked, percentile=99.0)
        im = ax.imshow(masked, origin="lower", cmap=wfe_cmap, vmin=vmin, vmax=vmax)
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])
        plt.colorbar(im, ax=ax, shrink=0.75)

    labels = list(item["stored_low_order_coefficients_nm"]["truth"].keys())
    x = np.arange(len(labels))
    stored = item["stored_low_order_coefficients_nm"]
    axes.flat[6].bar(x - 0.25, [stored["truth"][k] for k in labels], width=0.25, label="truth")
    axes.flat[6].bar(x, [stored["inference"][k] for k in labels], width=0.25, label="inference")
    axes.flat[6].bar(x + 0.25, [stored["error"][k] for k in labels], width=0.25, label="error")
    axes.flat[6].set_xticks(x, labels, rotation=45)
    axes.flat[6].set_title("Stored low-order coefficients [nm]")
    axes.flat[6].set_xlabel("Noll label; array index 0 maps to Z4")
    axes.flat[6].grid(True, axis="y", alpha=0.25)
    axes.flat[6].legend(fontsize=8)

    projection = item["residual_low_order_projection_nm"]
    axes.flat[7].bar(x - 0.25, [projection["truth_high_order_residual"][k] for k in labels], width=0.25, label="truth residual")
    axes.flat[7].bar(x, [projection["knowledge_error_residual"][k] for k in labels], width=0.25, label="knowledge error")
    axes.flat[7].bar(x + 0.25, [projection["inference_high_order"][k] for k in labels], width=0.25, label="inference")
    axes.flat[7].set_xticks(x, labels, rotation=45)
    axes.flat[7].set_title("Residual low-order projection of high-order maps [nm]")
    axes.flat[7].grid(True, axis="y", alpha=0.25)
    axes.flat[7].legend(fontsize=8)
    plt.tight_layout()
    plt.show()


The residual projection bars are expected to be near zero. Differences at the numerical-noise level do not represent meaningful low-order WFE bias. Meaningful low-order bias should be read from the stored low-order coefficient plot/table.


## 5. Optics and system-preset review


In [ ]:
optics_rows = pd.DataFrame(review.optics_diff_table(base_system, truth_system, inference_system))
print("Base optics")
print(json.dumps(review.summarize_optics_config(base_system), indent=2, default=str))
print("Truth optics")
print(json.dumps(review.summarize_optics_config(truth_system), indent=2, default=str))
print("Inference optics")
print(json.dumps(review.summarize_optics_config(inference_system), indent=2, default=str))
notebook_display(optics_rows)


## 6. Detector layer and calibration-map review

Absent maps are reported clearly rather than treated as errors.


In [ ]:
det_truth = review.summarize_detector_config(truth_system)
det_inf = review.summarize_detector_config(inference_system)
print("truth detector:")
print(json.dumps(det_truth, indent=2, default=str))
print("inference detector:")
print(json.dumps(det_inf, indent=2, default=str))
notebook_display(pd.DataFrame(det_truth["layers"]))
cal_maps = review.load_detector_calibration_maps(truth_system)
print("loaded calibration maps:", list(cal_maps))
if not cal_maps:
    print("No detector calibration maps loaded or implemented for this smoke path.")


In [ ]:
if cal_maps:
    n = len(cal_maps)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4), squeeze=False)
    for ax, (name, arr) in zip(axes.flat, cal_maps.items()):
        im = ax.imshow(arr, origin="lower")
        ax.set_title(name)
        plt.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()


## 7. Campaign noise-path audit

This section audits render/data noise separately from the inference likelihood variance model. It renders the resolved truth system through the same Binder path as the full-fidelity campaign review, using the configured PSF size with a minimum review render size. Display crops, when enabled, are applied only after the full image and variance maps are rendered.

The noise review resolves exposure time from the campaign/subblock configuration and prints its provenance below. A high colorbar value is meaningful only after checking that exposure-time provenance and image units match the render convention.

In [ ]:
noise_summary = review.summarize_noise_config(translated, truth_system)
noise_render = None
if ENABLE_NOISE_AUDIT:
    noise_render = review.render_noise_review_images(
        translated,
        truth_system,
        seed=123,
        min_psf_npix=NOISE_REVIEW_MIN_PSF_NPIX,
        default_psf_npix=NOISE_REVIEW_DEFAULT_PSF_NPIX,
        display_crop_npix=NOISE_REVIEW_DISPLAY_CROP_NPIX,
    )
    noise_summary["variance_diagnostics"] = noise_render["variance_diagnostics"]
    noise_summary["render_noise"] = noise_render["render_noise"]
    noise_summary["psf_npix_provenance"] = noise_render["psf_npix_provenance"]
    noise_summary["exposure_time_s_provenance"] = noise_render["exposure_time_s_provenance"]
    noise_summary["render_shape"] = noise_render["render_shape"]
    noise_summary["display_shape"] = noise_render["display_shape"]
    noise_summary["warnings"] = list(noise_summary.get("warnings", [])) + list(noise_render.get("warnings", []))
    print(f"Noise render shape: {noise_render['render_shape']}")
    print(f"Display crop shape: {noise_render['display_shape']}")
    print(f"psf_npix source: {noise_render['psf_npix_provenance']['source_field_path']}")
    print(f"exposure_time_s source: {noise_render['exposure_time_s_provenance']['source_field_path']}")
print(json.dumps(noise_summary, indent=2, default=str))


In [ ]:
if noise_render is not None and noise_render.get("available"):
    display_images = noise_render["display"]
    rn = noise_summary["render_noise"]
    vd = noise_summary["variance_diagnostics"]
    fig, axes = plt.subplots(2, 4, figsize=(16, 7))
    panels = [
        (f"Noiseless resolved-system image\npsf_npix={rn['rendered_psf_npix']}, exposure={rn['exposure_time_s']} s", display_images["noiseless"], "electrons"),
        (f"Configured noisy image\nshot={rn['shot_noise']}, read={rn['read_noise']}, dark={rn['dark_current']}", display_images["configured_noisy"], "electrons"),
        ("Noise residual", display_images["noise_residual"], "electrons"),
        ("Expected variance", display_images["expected_variance"], "electrons^2"),
    ]
    for ax, (title, image, units) in zip(axes[0], panels):
        im = ax.imshow(image, origin="lower")
        ax.set_title(title)
        cb = plt.colorbar(im, ax=ax, shrink=0.75)
        cb.set_label(units)

    im = axes[1, 0].imshow(display_images["normalized_residual"], origin="lower", vmin=-5, vmax=5)
    axes[1, 0].set_title(f"Residual / sqrt(expected variance)\nread noise={rn['read_noise_electrons']} e-, variance_floor={noise_summary['inference_noise_model']['variance_floor']}")
    cb = plt.colorbar(im, ax=axes[1, 0], shrink=0.75)
    cb.set_label("sigma")
    axes[1, 1].hist(display_images["noise_residual"].ravel(), bins=50)
    axes[1, 1].set_title("Residual histogram")
    axes[1, 2].hist(display_images["normalized_residual"].ravel(), bins=50, range=(-5, 5))
    axes[1, 2].set_title("Normalized residual histogram")
    axes[1, 3].axis("off")
    inf = noise_summary["inference_noise_model"]
    lines = [
        f"rendered_psf_npix: {rn['rendered_psf_npix']}",
        f"displayed_crop_npix: {rn['displayed_crop_npix']}",
        f"render shape: {rn['render_shape']}",
        f"display shape: {rn['display_shape']}",
        f"exposure: {rn['exposure_time_s']} s ({rn['exposure_time_s_source']})",
        f"model sum: {vd['model_image_sum']:.6g}",
        f"model peak: {vd['model_image_peak']:.6g}",
        f"photon variance peak: {vd['expected_photon_variance_peak']:.6g}",
        f"read noise: {rn['read_noise_electrons']} e- ({rn['read_noise_source']})",
        f"dark current: {rn['dark_current_e_per_s']} e-/s ({rn['dark_current_source']})",
        f"inference variance: {inf['variance_model']}",
    ]
    axes[1, 3].text(0.0, 1.0, "\n".join(lines), va="top", family="monospace", fontsize=8)
    plt.tight_layout()
    plt.show()


## 8. Trajectory filtering and subblock timing review

The trajectory trace source defines the continuous pointing/PA time series. The subblock planner selects discrete frame times from that series. The iterative planner groups generated subblocks into update windows. These are related but distinct pieces of configuration.

The selected-segment panels show frame centers used by each subblock. If `n_frames` is small and subblocks begin at one-second intervals, the plotted frame samples will appear as separated clusters rather than a continuous trajectory.


In [ ]:
from IPython.display import display as notebook_display

trajectory_review = review.load_trajectory_for_review(translated)
hp_review = review.make_high_pass_trajectory_diagnostic(trajectory_review, timescale_s=15.0)
filter_prov = trajectory_review.get("summary", {}).get("filter", {})

print(json.dumps(trajectory_review.get("summary", trajectory_review), indent=2, default=str))
print("configured filter enabled:", filter_prov.get("enabled", False))
print("configured filter kind:", filter_prov.get("kind"))
print("moving-average diagnostic note:", hp_review.get("note"))

if trajectory_review.get("warnings"):
    print("trajectory/subblock warnings:")
    for warning in trajectory_review["warnings"]:
        print("-", warning)

if trajectory_review.get("available"):
    plan_df = pd.DataFrame([trajectory_review.get("subblock_plan", {})])
    notebook_display(plan_df[[
        "subblocks_n_subblocks",
        "trace_source_window_n_subblocks",
        "iterative_windows_per_draw",
        "iterative_subblocks_per_window",
        "expected_iterative_subblocks",
        "resolved_n_subblocks",
        "consistency_status",
        "canonical_source",
    ]])

    timing_df = pd.DataFrame(review.trajectory_timing_summary_table(trajectory_review))
    notebook_display(timing_df)

    filter_df = pd.DataFrame(review.trajectory_filter_provenance_table(trajectory_review))
    notebook_display(filter_df[[
        "key",
        "filter_enabled",
        "filter_kind",
        "method",
        "order",
        "cutoff_period_s",
        "cutoff_hz",
        "raw_rms",
        "filtered_rms",
        "removed_rms",
        "selected_window_raw_rms",
        "selected_window_filtered_rms",
        "removed_component_label",
    ]])


In [ ]:
from IPython.display import display as notebook_display

if trajectory_review.get("available"):
    figures = review.plot_trajectory_review_components(trajectory_review)
    for fig in figures:
        notebook_display(fig)
        plt.close(fig)


In [ ]:
if trajectory_review.get("available"):
    filter_df = pd.DataFrame(review.trajectory_filter_provenance_table(trajectory_review))
    if not filter_df.empty:
        fig, ax = plt.subplots(figsize=(10, 4))
        filter_df.set_index("key")[["raw_rms", "filtered_rms", "removed_rms"]].plot(kind="bar", ax=ax)
        ax.set_title("trajectory RMS by raw, filtered, and explicitly removed components")
        ax.set_ylabel("RMS")
        ax.grid(True, axis="y", alpha=0.25)
        plt.tight_layout()
        plt.show()


In [ ]:
if trajectory_review.get("available") and filter_prov.get("frequency_response"):
    response = filter_prov["frequency_response"]
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(response.get("frequency_hz", []), response.get("gain", []))
    ax.set_xlabel("frequency [Hz]")
    ax.set_ylabel("gain")
    ax.set_title("configured Bessel filter response")
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

if trajectory_review.get("available"):
    traj = trajectory_review["trajectory"]
    dt = float(np.median(np.diff(traj.time_s)))
    fig, axes = plt.subplots(len(traj.values), 1, figsize=(11, 3 * len(traj.values)), squeeze=False)
    for ax, key in zip(axes[:, 0], traj.values):
        raw = np.asarray((traj.unfiltered_values or traj.values)[key], dtype=float)
        filt = np.asarray(traj.values[key], dtype=float)
        freq = np.fft.rfftfreq(raw.size, d=dt)
        ax.semilogy(freq[1:], np.abs(np.fft.rfft(raw - raw.mean()))[1:] ** 2, label="raw")
        ax.semilogy(freq[1:], np.abs(np.fft.rfft(filt - filt.mean()))[1:] ** 2, label="filtered")
        ax.set_title(f"FFT power {key}")
        ax.set_xlabel("frequency [Hz]")
        ax.grid(True, alpha=0.25)
        ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
if trajectory_review.get("available") and hp_review.get("available"):
    fig, axes = plt.subplots(len(hp_review["series"]), 1, figsize=(12, 3 * len(hp_review["series"])), squeeze=False)
    for ax, key in zip(axes[:, 0], hp_review["series"]):
        series = hp_review["series"][key]
        ax.plot(trajectory_review["trajectory"].time_s, series["raw"], label="raw", alpha=0.5)
        ax.plot(trajectory_review["trajectory"].time_s, series["low_pass"], label="15 s moving average")
        ax.plot(trajectory_review["trajectory"].time_s, series["high_pass"], label="diagnostic residual")
        ax.set_title(f"moving-average diagnostic {key}; residual RMS={series['rms_high_pass']:.4g}")
        ax.grid(True, alpha=0.25)
        ax.legend()
    plt.tight_layout()
    plt.show()


## 9. Trace jitter review

Explicit question: does `trace_jitter` add noise on top of the trajectory?


In [ ]:
from IPython.display import display as notebook_display

trace_jitter_review = review.compare_trace_jitter_enabled_disabled(translated)
print(json.dumps(trace_jitter_review, indent=2, default=str))
notebook_display(pd.DataFrame([{"key": k, "rms_difference": v} for k, v in trace_jitter_review.get("rms_difference", {}).items()]))


## 10. Model rendering sanity checks

The campaign noise-path audit above performs the lightweight resolved-system Binder render used for sanity checks. This cell reports the render status without launching a campaign.


In [ ]:
render_review = {
    "available": bool(noise_render is not None and noise_render.get("available")),
    "source": None if noise_render is None else noise_render.get("source"),
    "variance_diagnostics": None if noise_render is None else noise_render.get("variance_diagnostics"),
}
print(json.dumps(render_review, indent=2, default=str))


## 11. Summary dashboard


In [ ]:
from IPython.display import display as notebook_display

dashboard = pd.DataFrame(review.summary_dashboard(
    config=translated,
    base_cfg=base_system,
    truth_cfg=truth_system,
    inference_cfg=inference_system,
    model_split=split,
    trajectory_review=trajectory_review,
    trace_jitter_review=trace_jitter_review,
))
notebook_display(dashboard)


In [ ]:
if WRITE_ARTIFACTS:
    artifact_paths = review.write_review_artifacts(
        outdir,
        base_system_cfg=base_system,
        truth_system_cfg=truth_system,
        inference_system_cfg=inference_system,
        model_split=split,
        spectral_summary=spectral_summary,
        wfe_summary=wfe_summary,
        detector_summary=det_truth,
        noise_summary=noise_summary,
        trajectory_summary=trajectory_review.get("summary", trajectory_review),
    )
    print(json.dumps(artifact_paths, indent=2))


## Reviewer notes / decisions

- Source / spectral deck decision:
- WFE map decision:
- Detector / calibration decision:
- Noise decision:
- Trajectory / high-pass decision:
- Trace jitter decision:
- Config changes to make before campaign:
- Follow-up tasks:
